In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

# 解决中文、负号显示问题
plt.rcParams['font.sans-serif']=['SimHei']
plt.rcParams['axes.unicode_minus']=False

df_raw = pd.read_csv("../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv")
df = df_raw.copy()

print('\n----------前5行----------')
display(df.head())


----------前5行----------


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [2]:
print(f"原始数据集形状:{df.shape}")
print('\n----------数据类型信息----------')
df.info()

原始数据集形状:(7043, 21)

----------数据类型信息----------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 


In [3]:
# 1. 检查TotalCharges字段
# 1.1 查看TotalCharges字段的数据类型
print("TotalCharges数据类型:",df['TotalCharges'].dtype)
# 1.2 查看TotalCharges字段等于空字符串的行
empty_total = df[df['TotalCharges']==' ']
print(f"TotalCharges为空字符串的样本数量:{len(empty_total)}")
display(empty_total[['customerID','tenure','MonthlyCharges','TotalCharges']])
# 1.3 查看是否存在完全重复的行
print("完全重复行数:",df.duplicated().sum())
print("customerID是否唯一:",df['customerID'].nunique()==len(df))

TotalCharges数据类型: object
TotalCharges为空字符串的样本数量:11


,customerID,tenure,MonthlyCharges,TotalCharges
488,4472-LVYGI,0,52.55,
753,3115-CZMZD,0,20.25,
936,5709-LVOEQ,0,80.85,
1082,4367-NUYAO,0,25.75,
1340,1371-DWPAZ,0,56.05,
3331,7644-OMVMY,0,19.85,
3826,3213-VVOLG,0,25.35,
4380,2520-SGTTA,0,20.00,
5218,2923-ARZLG,0,19.70,
6670,4075-WKNIU,0,73.35,


完全重复行数: 0
customerID是否唯一: True


In [4]:
# 2. 处理 TotalCharges
df_clean = df.copy()
df_clean['TotalCharges'] = df_clean['TotalCharges'].replace(' ','0')
df_clean['TotalCharges'] = df_clean['TotalCharges'].astype(float)
# 校验
print("TotalCharges数据类型:",df_clean['TotalCharges'].dtype)
print("转换后TotalCharges等于0的样本数量:",(df_clean['TotalCharges']==0).sum())
display(df_clean[df_clean['TotalCharges']==0][['customerID','tenure','MonthlyCharges','TotalCharges']])

TotalCharges数据类型: float64
转换后TotalCharges等于0的样本数量: 11


,customerID,tenure,MonthlyCharges,TotalCharges
488,4472-LVYGI,0,52.55,0.0
753,3115-CZMZD,0,20.25,0.0
936,5709-LVOEQ,0,80.85,0.0
1082,4367-NUYAO,0,25.75,0.0
1340,1371-DWPAZ,0,56.05,0.0
3331,7644-OMVMY,0,19.85,0.0
3826,3213-VVOLG,0,25.35,0.0
4380,2520-SGTTA,0,20.00,0.0
5218,2923-ARZLG,0,19.70,0.0
6670,4075-WKNIU,0,73.35,0.0


In [5]:
# 3. 标签编码 + 二分类特征 0-1 编码
df_encoded = df_clean.copy()
# 3.1 目标标签编码，Yes→1，No→0
df_encoded['Churn'] = df_encoded['Churn'].map({'Yes':1,'No':0})
print("Churn标签分布:",df_encoded['Churn'].value_counts())

# 3.2 二分类特征 0-1 编码
binary_cols = ['gender','Partner','Dependents','PhoneService','PaperlessBilling']
for col in binary_cols:
    if col == 'gender':
        df_encoded[col] = df_encoded[col].map({'Female':1,'Male':0})
    else:
        df_encoded[col] = df_encoded[col].map({'Yes':1,'No':0})
print('\n----------二分类特征编码完成，查看样例----------')
display(df_encoded[binary_cols].head())

Churn标签分布: Churn
0    5174
1    1869
Name: count, dtype: int64

----------二分类特征编码完成，查看样例----------


,gender,Partner,Dependents,PhoneService,PaperlessBilling
0,1,1,0,0,1
1,0,0,0,1,0
2,0,0,0,1,1
3,0,0,0,0,0
4,1,0,0,1,1


In [9]:
# 4. 多分类特征 OneHot 编码
df_onehot = df_encoded.copy()
multi_cols = [
    'MultipleLines',
    'InternetService',
    'OnlineSecurity',
    'OnlineBackup',
    'DeviceProtection',
    'TechSupport',
    'StreamingTV',
    'StreamingMovies',
    'Contract',
    'PaymentMethod'
    ]
df_onehot = pd.get_dummies(df_onehot,columns=multi_cols,drop_first=False,dtype=int)
print("onehot后的数据集形状:",df_onehot.shape)
display(df_onehot.head())
print('\n'+'-'*10+'查看新增列名'+'-'*10)
for c in df_onehot.columns:
    print(c)


onehot后的数据集形状: (7043, 42)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,PaperlessBilling,MonthlyCharges,TotalCharges,...,StreamingMovies_No,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_Month-to-month,Contract_One year,Contract_Two year,PaymentMethod_Bank transfer (automatic),PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,7590-VHVEG,1,0,1,0,1,0,1,29.85,29.85,...,1,0,0,1,0,0,0,0,1,0
1,5575-GNVDE,0,0,0,0,34,1,0,56.95,1889.50,...,1,0,0,0,1,0,0,0,0,1
2,3668-QPYBK,0,0,0,0,2,1,1,53.85,108.15,...,1,0,0,1,0,0,0,0,0,1
3,7795-CFOCW,0,0,0,0,45,0,0,42.30,1840.75,...,1,0,0,0,1,0,1,0,0,0
4,9237-HQITU,1,0,0,0,2,1,1,70.70,151.65,...,1,0,0,1,0,0,0,0,1,0



----------查看新增列名----------
customerID
gender
SeniorCitizen
Partner
Dependents
tenure
PhoneService
PaperlessBilling
MonthlyCharges
TotalCharges
Churn
MultipleLines_No
MultipleLines_No phone service
MultipleLines_Yes
InternetService_DSL
InternetService_Fiber optic
InternetService_No
OnlineSecurity_No
OnlineSecurity_No internet service
OnlineSecurity_Yes
OnlineBackup_No
OnlineBackup_No internet service
OnlineBackup_Yes
DeviceProtection_No
DeviceProtection_No internet service
DeviceProtection_Yes
TechSupport_No
TechSupport_No internet service
TechSupport_Yes
StreamingTV_No
StreamingTV_No internet service
StreamingTV_Yes
StreamingMovies_No
StreamingMovies_No internet service
StreamingMovies_Yes
Contract_Month-to-month
Contract_One year
Contract_Two year
PaymentMethod_Bank transfer (automatic)
PaymentMethod_Credit card (automatic)
PaymentMethod_Electronic check
PaymentMethod_Mailed check


In [13]:
# 5. 业务衍生特征
df_feature = df_onehot.copy()
# 5.1 月均消费
df_feature['avg_monthly_spend'] = np.where(
    df_feature['tenure'] == 0,
    df_feature['MonthlyCharges'],
    df_feature['TotalCharges'] / df_feature['tenure']
)
# 5.2 用户生命周期分组
max_tenure = df_feature['tenure'].max() 
bins = [-1, 6, 24, max_tenure]
df_feature['user_life']=pd.cut(
    df_feature['tenure'],
    bins = bins,
    labels = ["new_user", "mid_user", "long_user"],
    right = True
)
# 5.3 单月消费占比
df_feature['charge_ratio'] = df_feature['MonthlyCharges'] / df_feature["TotalCharges"].replace(0, np.nan)

print('\n'+'-'*10+'查看衍生特征样例'+'-'*10)
display(df_feature[["tenure","MonthlyCharges","TotalCharges","avg_monthly_spend","user_life","charge_ratio"]].head(10))


----------查看衍生特征样例----------


,tenure,MonthlyCharges,TotalCharges,avg_monthly_spend,user_life,charge_ratio
0,1,29.85,29.85,29.850000,new_user,1.000000
1,34,56.95,1889.50,55.573529,long_user,0.030140
2,2,53.85,108.15,54.075000,new_user,0.497920
3,45,42.30,1840.75,40.905556,long_user,0.022980
4,2,70.70,151.65,75.825000,new_user,0.466205
5,8,99.65,820.50,102.562500,mid_user,0.121450
6,22,89.10,1949.40,88.609091,mid_user,0.045706
7,10,29.75,301.90,30.190000,mid_user,0.098543
8,28,104.80,3046.05,108.787500,long_user,0.034405
9,62,56.15,3487.95,56.257258,long_user,0.016098


In [14]:
# 6. 清理无关字段，保存预处理数据集
df_final = df_feature.drop(columns=['customerID'])
print("预处理后数据集形状:",df_final.shape)
display(df_final.head(10))
# 保存到output/processed下
df_final.to_csv('../data/processed/churn_processed.csv',index=False,encoding='utf-8')
print('\n'+'-'*10+'预处理数据集已保存'+'-'*10)

预处理后数据集形状: (7043, 44)


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,PaperlessBilling,MonthlyCharges,TotalCharges,Churn,...,Contract_Month-to-month,Contract_One year,Contract_Two year,PaymentMethod_Bank transfer (automatic),PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,avg_monthly_spend,user_life,charge_ratio
0,1,0,1,0,1,0,1,29.85,29.85,0,...,1,0,0,0,0,1,0,29.850000,new_user,1.000000
1,0,0,0,0,34,1,0,56.95,1889.50,0,...,0,1,0,0,0,0,1,55.573529,long_user,0.030140
2,0,0,0,0,2,1,1,53.85,108.15,1,...,1,0,0,0,0,0,1,54.075000,new_user,0.497920
3,0,0,0,0,45,0,0,42.30,1840.75,0,...,0,1,0,1,0,0,0,40.905556,long_user,0.022980
4,1,0,0,0,2,1,1,70.70,151.65,1,...,1,0,0,0,0,1,0,75.825000,new_user,0.466205
5,1,0,0,0,8,1,1,99.65,820.50,1,...,1,0,0,0,0,1,0,102.562500,mid_user,0.121450
6,0,0,0,1,22,1,1,89.10,1949.40,0,...,1,0,0,0,1,0,0,88.609091,mid_user,0.045706
7,1,0,0,0,10,0,0,29.75,301.90,0,...,1,0,0,0,0,0,1,30.190000,mid_user,0.098543
8,1,0,1,0,28,1,1,104.80,3046.05,1,...,1,0,0,0,0,1,0,108.787500,long_user,0.034405
9,0,0,0,1,62,1,0,56.15,3487.95,0,...,0,1,0,1,0,0,0,56.257258,long_user,0.016098



----------预处理数据集已保存----------
